## Imports

In [0]:
from pyspark.sql import SparkSession
import pandas as pd

## Get the needed Parameters

In [0]:
run_date = dbutils.widgets.get("run_date")

Businessdate = spark.sql(f"""                    
                        select 
                            case 
                                when try_cast({run_date!r} as date) is not null then try_cast({run_date!r} as date)
                                else PreviousBusinessDay
                            end as BusinessDate
                        from `p-cdh`.reference.ref_calendar
                        where CalendarDate = coalesce(try_cast({run_date!r} as date),cast(getdate() as date))
                    """).first()[0]

print(Businessdate)

## Get the needed Source Tables

In [0]:
Portfolio = spark.sql(f"""                   
                            SELECT 625244    AS portfolio_id_thesis
                                 , 'DMW'     AS Prefix
                            UNION          
                            SELECT 625243    AS portfolio_id_thesis
                                 , 'DMR'     AS Prefix
                            UNION          
                            SELECT NULL      AS portfolio_id_thesis
                                 , 'DMD'     AS Prefix
                        """)

Portfolio.createOrReplaceTempView("Portfolio")
display(Portfolio)

### Get Profits Source Tables

In [0]:
loan_details = spark.sql(f"""                    
                            select distinct agreement_number, Account_Number
                            from `p-cdh`.landed.profits_loan_details
                            where BusinessDate = '{Businessdate}'
                        """)

loan_details.createOrReplaceTempView("loan_details")
# display(loan_details)

In [0]:
customer = spark.sql(f"""                    
                        select distinct cdi
                        from `p-cdh`.landed.profits_customer
                        where BusinessDate = '{Businessdate}'
                    """)

customer.createOrReplaceTempView("customer")
# display(customer)

In [0]:
agreement = spark.sql(f"""                    
                        select distinct agreement_number
                        from `p-cdh`.landed.profits_agreement
                        where BusinessDate = '{Businessdate}'
                    """)

agreement.createOrReplaceTempView("agreement")
# display(customer)

### Get Thesis Source Tables

In [0]:
prperson = spark.sql(f"""                    
                        select distinct crs
                        from `p-cdh`.landed.thesis_prperson
                        where BusinessDate = '{Businessdate}'
                    """)

prperson.createOrReplaceTempView("prperson")
# display(prperson)

In [0]:
prproperty = spark.sql(f"""                    
                        select distinct portfolio_code, id
                        from `p-cdh`.landed.thesis_prproperty
                        where BusinessDate = '{Businessdate}'
                    """)

prproperty.createOrReplaceTempView("prproperty")
# display(prproperty)

In [0]:
pragreement = spark.sql(f"""                    
                            select distinct org_contract_id, id, reference_key
                            from `p-cdh`.landed.thesis_pragreement
                            where BusinessDate = '{Businessdate}'
                        """)

pragreement.createOrReplaceTempView("pragreement")
# display(pragreement)

In [0]:
pragreement_loan = spark.sql(f"""                    
                                select distinct account_no, agreement_id, reference_key
                                from `p-cdh`.landed.thesis_pragreement_loan
                                where BusinessDate = '{Businessdate}'
                            """)

pragreement_loan.createOrReplaceTempView("pragreement_loan")
# display(pragreement_loan)

In [0]:
prcadastre = spark.sql(f"""                    
                            select distinct portfolio_code, id
                            from `p-cdh`.landed.thesis_prcadastre
                            where BusinessDate = '{Businessdate}'
                        """)

prcadastre.createOrReplaceTempView("prcadastre")
# display(prcadastre)

In [0]:
prcollateral = spark.sql(f"""                    
                            select distinct id, portfolio_owner, reference_key
                            from `p-cdh`.landed.thesis_prcollateral
                            where BusinessDate = '{Businessdate}'
                        """)

prcollateral.createOrReplaceTempView("prcollateral")
display(prcollateral)

In [0]:
prcollateral_re = spark.sql(f"""                    
                            select distinct collateral_id
                            from `p-cdh`.landed.thesis_prcollateral_re
                            where BusinessDate = '{Businessdate}'
                        """)

prcollateral_re.createOrReplaceTempView("prcollateral_re")
# display(prcollateral_re)

In [0]:
prcollateraldet = spark.sql(f"""                    
                            select distinct collateral_id
                            from `p-cdh`.landed.thesis_prcollateraldet
                            where BusinessDate = '{Businessdate}'
                        """)

prcollateraldet.createOrReplaceTempView("prcollateraldet")
# display(prcollateraldet)

In [0]:
prproperty_owner = spark.sql(f"""                    
                                select distinct property_id
                                from `p-cdh`.landed.thesis_prproperty_owner
                                where BusinessDate = '{Businessdate}'
                            """)

prproperty_owner.createOrReplaceTempView("prproperty_owner")
# display(prproperty_owner)

In [0]:
pragreement_collateral = spark.sql(f"""                    
                                        select distinct collateral_id, collateral_code
                                        from `p-cdh`.landed.thesis_pragreement_collateral
                                        where BusinessDate = '{Businessdate}'
                                    """)

pragreement_collateral.createOrReplaceTempView("pragreement_collateral")
display(pragreement_collateral)

In [0]:
pragreement_collateral = spark.sql(f"""                    
                                        select distinct collateral_id, collateral_code
                                        from `p-cdh`.landed.thesis_pragreement_collateral
                                        where BusinessDate = '{Businessdate}'
                                    """)

pragreement_collateral.createOrReplaceTempView("pragreement_collateral")
# display(pragreement_collateral)

In [0]:
pragreement_loan = spark.sql(f"""                    
                                select distinct reference_key, account_no, agreement_id
                                from `p-cdh`.landed.thesis_pragreement_loan
                                where BusinessDate = '{Businessdate}'
                            """)

pragreement_loan.createOrReplaceTempView("pragreement_loan")
# display(pragreement_loan)

## Check Tables

### Agreements - Accounts

#### Check in pragreement

In [0]:
NumberOfAgreements = spark.sql(f"""                    
                                select distinct port.Prefix, count(distinct org_contract_id)
                                from pragreement agr
                                inner join Portfolio port
                                on port.Prefix = left(reference_key,3)
                                group by all
                            """)

NumberOfAgreements.createOrReplaceTempView("NumberOfAgreements")
display(NumberOfAgreements)

In [0]:
ComparisonOfAgreements = spark.sql(f""" 
                                        -- select org_contract_id
                                        -- from pragreement
                                        -- inner join Portfolio port
                                        -- on port.Prefix = left(org_contract_id,3)
                                        -- except
                                        select agreement_number
                                        from loan_details
                                        inner join Portfolio port
                                        on port.Prefix = left(agreement_number,3)
                                        except
                                        select org_contract_id
                                        from pragreement
                                        inner join Portfolio port
                                        on port.Prefix = left(org_contract_id,3)
                                    """)

ComparisonOfAgreements.createOrReplaceTempView("ComparisonOfAgreements")
display(ComparisonOfAgreements)

#### Check in pragreement_loan

In [0]:
ComparisonOfAccounts = spark.sql(f""" 
                                    select Account_Number
                                    from loan_details
                                    inner join Portfolio port
                                    on port.Prefix = left(Account_Number,3)
                                    except
                                    select Account_No
                                    from pragreement_loan
                                    inner join Portfolio port
                                    on port.Prefix = left(Account_No,3)
                                    -- except
                                    -- select Account_Number
                                    -- from loan_details
                                    -- inner join Portfolio port
                                    -- on port.Prefix = left(Account_Number,3)
                                """)

ComparisonOfAccounts.createOrReplaceTempView("ComparisonOfAccounts")
display(ComparisonOfAccounts)

In [0]:
ComparisonOfAgreements = spark.sql(f""" 
                                        select Agreement_Number
                                        from agreement
                                        inner join Portfolio port
                                        on port.Prefix = left(Agreement_Number,3)
                                        except
                                        select org_contract_id
                                        from pragreement_loan agrloan
                                        inner join pragreement agr
                                        on agrloan.agreement_id = agr.id
                                        inner join Portfolio port
                                        on port.Prefix = left(org_contract_id,3)
                                        -- except
                                        -- select Agreement_Number
                                        -- from agreement
                                        -- inner join Portfolio port
                                        -- on port.Prefix = left(Agreement_Number,3)
                                    """)

ComparisonOfAgreements.createOrReplaceTempView("ComparisonOfAgreements")
display(ComparisonOfAgreements)

### Customers

#### Check in prperson

In [0]:
NumberOfPersons = spark.sql(f"""                    
                                select distinct port.Prefix, count(distinct crs)
                                from prperson prp
                                inner join Portfolio port
                                on port.Prefix = left(prp.crs,3)
                                group by all
                            """)

NumberOfPersons.createOrReplaceTempView("NumberOfPersons")
display(NumberOfPersons)

In [0]:
ComparisonOfPersons = spark.sql(f"""                    
                                    select port.Prefix, crs
                                    from prperson prp
                                    inner join Portfolio port
                                    on port.Prefix = left(prp.crs,3)
                                    except
                                    select port.Prefix, cdi
                                    from customer cust
                                    inner join Portfolio port
                                    on port.Prefix = left(cust.cdi,3)
                                """)

ComparisonOfPersons.createOrReplaceTempView("ComparisonOfPersons")
display(ComparisonOfPersons)

### Collateral

#### Check in prcollateral

In [0]:
Right_Portfolio_Owner = spark.sql(f"""                    
                                    select distinct port.Prefix, portfolio_owner
                                    from prcollateral
                                    inner join Portfolio port
                                    on port.Prefix = left(reference_key, 3)
                                    group by all
                                """)

Right_Portfolio_Owner.createOrReplaceTempView("Right_Portfolio_Owner")
display(Right_Portfolio_Owner)

In [0]:
Right_Prefix = spark.sql(f"""                    
                            select distinct port.Prefix, portfolio_owner
                            from prcollateral
                            inner join Portfolio port
                            on portfolio_id_thesis = portfolio_owner
                            group by all
                        """)

Right_Prefix.createOrReplaceTempView("Right_Prefix")
display(Right_Prefix)

In [0]:
NumberOfCollateral = spark.sql(f"""                    
                                    select distinct port.Prefix, count(distinct id) countAll
                                    from prcollateral col
                                    inner join Portfolio port
                                    on port.Prefix = left(reference_key, 3)
                                    group by all
                                """)

NumberOfCollateral.createOrReplaceTempView("NumberOfCollateral")
display(NumberOfCollateral)

#### Check in prcollateral_re

In [0]:
NumberOfCollateral_re = spark.sql(f"""                    
                                      select distinct port.Prefix, count(distinct re.collateral_id) countRE
                                      from prcollateral_re re
                                      inner join prcollateral col
                                      on col.id = re.collateral_id
                                      inner join Portfolio port
                                      on port.Prefix = left(reference_key, 3)
                                      group by all
                                  """)

NumberOfCollateral_re.createOrReplaceTempView("NumberOfCollateral_re")
display(NumberOfCollateral_re)

#### Check in prcollateraldet

In [0]:
NumberOfCollateralDet = spark.sql(f"""                    
                                    select distinct port.Prefix, count(distinct reference_key) countDet
                                    from prcollateraldet det
                                    inner join prcollateral col
                                    on col.id = det.collateral_id
                                    inner join Portfolio port
                                    on port.Prefix = left(reference_key, 3)
                                    group by all
                                """)

NumberOfCollateralDet.createOrReplaceTempView("NumberOfCollateralDet")
display(NumberOfCollateralDet)

#### Check the Total Collateral

In [0]:
NumberOfAllCollaterals = spark.sql(f"""                    
                                    select distinct col.Prefix, countRE, countDet, (countRE + countDet) Sum, countAll
                                    from NumberOfCollateral col
                                    inner join NumberOfCollateral_re re
                                    on col.Prefix = re.Prefix
                                    inner join NumberOfCollateralDet det
                                    on col.Prefix = det.Prefix
                                """)

NumberOfAllCollaterals.createOrReplaceTempView("NumberOfAllCollaterals")
display(NumberOfAllCollaterals)

#### Check in pragreement_collateral

In [0]:
AgreementXCollateralRltp = spark.sql(f"""                    
                                        select distinct port.Prefix, count(distinct collateral_id) countRE
                                        from pragreement_collateral
                                        inner join Portfolio port
                                        on port.Prefix = left(collateral_code, 3)
                                        group by all
                                    """)

AgreementXCollateralRltp.createOrReplaceTempView("AgreementXCollateralRltp")
display(AgreementXCollateralRltp)

In [0]:
CollateralWithoutAgreement = spark.sql(f"""                    
                                        SELECT DISTINCT reference_key
                                        FROM prcollateral
                                        inner join Portfolio port
                                        on port.Prefix = left(reference_key, 3)
                                        EXCEPT 
                                        SELECT DISTINCT collateral_code
                                        FROM pragreement_collateral
                                        inner join Portfolio port
                                        on port.Prefix = left(collateral_code, 3)
                                    """)

CollateralWithoutAgreement.createOrReplaceTempView("CollateralWithoutAgreement")
display(CollateralWithoutAgreement)

### Property

#### Check in prproperty

In [0]:
NumberOfProperties = spark.sql(f"""                    
                                select distinct port.Prefix, count(distinct id)
                                from prproperty prop
                                inner join Portfolio port
                                on port.Prefix = left(portfolio_code,3)
                                group by all
                            """)

NumberOfProperties.createOrReplaceTempView("NumberOfProperties")
display(NumberOfProperties)

#### Check in prproperty_owner

In [0]:
NumberOfProperties = spark.sql(f"""                    
                                select distinct port.Prefix, count(distinct property_id)
                                from prproperty_owner po
                                inner join prproperty pp
                                on po.property_id = pp.id
                                inner join Portfolio port
                                on port.Prefix = left(portfolio_code, 3)
                                group by all
                            """)

NumberOfProperties.createOrReplaceTempView("NumberOfProperties")
display(NumberOfProperties)

### ---------------------------------

### Other

#### Check in prcadastre

In [0]:
NumberOfCadastre = spark.sql(f"""                    
                                select distinct port.Prefix, count(distinct id)
                                from prcadastre
                                inner join Portfolio port
                                on port.Prefix = portfolio_code
                                group by all
                            """)

NumberOfCadastre.createOrReplaceTempView("NumberOfCadastre")
display(NumberOfCadastre)